In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, field


@dataclass
class GeneratedRecommendation:

    title: str
    reason: str
    action: str
    priority: str = "medium"
    concept_ids: list[str] = field(
        default_factory=list
    )
    source_types: list[str] = field(
        default_factory=list
    )


@dataclass
class RecommendationGenerationResult:

    recommendations: list[GeneratedRecommendation]


class Recommender:

    def __init__(self, ai_service):

        self.ai_service = ai_service

    def generate(
        self,
        mastery: list[dict],
        recent_performance: list[dict],
        goals: list[str],
        recent_activity: list[dict],
        user_id: str | None = None,
        project_id: str | None = None,
    ) -> RecommendationGenerationResult:

        from app.ai.prompts import RECOMMENDATION_SYSTEM_PROMPT

        # ========================================================
        # SAFE CONTEXT COMPACTION
        # ========================================================

        def compact(value, limit: int) -> str:
            try:
                text = json.dumps(
                    value,
                    default=str,
                    ensure_ascii=False,
                )
            except Exception:
                text = str(value)

            if len(text) <= limit:
                return text

            return text[:limit] + "…"

        # Keep the prompt comfortably below Groq's input budget.
        prompt = f"""
    Generate useful learning recommendations.

    Goals:
    {compact(goals, 1200)}

    Mastery:
    {compact(mastery[:8], 3500)}

    Recent performance:
    {compact(recent_performance[:5], 2500)}

    Recent activity:
    {compact(recent_activity[:8], 2500)}

    Return ONLY valid JSON.

    Required format:
    {{
      "recommendations": [
        {{
          "title": "short recommendation title",
          "reason": "why this is recommended",
          "action": "specific learner action",
          "priority": "low",
          "concept_ids": [],
          "source_types": []
        }}
      ]
    }}
    """

        # ========================================================
        # SECONDARY PROMPT TRIM
        # ========================================================

        if (len(prompt) + 3) // 4 > 5000:

            prompt = f"""
    Generate useful learning recommendations.

    Goals:
    {compact(goals, 700)}

    Mastery:
    {compact(mastery[:4], 1800)}

    Recent performance:
    {compact(recent_performance[:3], 1200)}

    Recent activity:
    {compact(recent_activity[:3], 1000)}

    Return ONLY valid JSON with a recommendations array.

    Each recommendation must contain:
    title, reason, action, priority, concept_ids, source_types.
    """

        approx_tokens = (len(prompt) + 3) // 4

        if approx_tokens > 5000:
            raise ValueError(
                "Recommendation prompt exceeds the safe token budget "
                f"after trimming (~{approx_tokens} tokens)."
            )

        print(
            "[Recommender] "
            f"approx_prompt_tokens={approx_tokens} "
            f"user_id={user_id} "
            f"project_id={project_id}"
        )

        # ========================================================
        # AI GENERATION
        # ========================================================

        raw = self.ai_service.generate_text(
            prompt=prompt,
            system_instruction=RECOMMENDATION_SYSTEM_PROMPT,
            operation="recommendation_generation",
            user_id=user_id,
            project_id=project_id,
        )

        if not raw or not raw.strip():
            raise RuntimeError(
                "Recommendation model returned an empty response."
            )

        # ========================================================
        # JSON PARSING
        # ========================================================

        try:
            json_text = self._extract_json(raw)

            data = json.loads(json_text)

        except json.JSONDecodeError as exc:
            print(
                "[Recommender] Invalid JSON returned by AI: "
                f"{exc}"
            )

            # Do not let malformed model output become a mysterious 500.
            raise RuntimeError(
                "Recommendation model returned invalid JSON."
            ) from exc

        except ValueError:
            raise

        # ========================================================
        # RESPONSE SHAPE VALIDATION
        # ========================================================

        recommendations_data = data.get(
            "recommendations",
            [],
        )

        if recommendations_data is None:
            recommendations_data = []

        if not isinstance(
            recommendations_data,
            list,
        ):
            raise RuntimeError(
                "Recommendation model returned an invalid "
                "'recommendations' field."
            )

        recommendations = []

        # ========================================================
        # NORMALIZE EACH RECOMMENDATION
        # ========================================================

        for item in recommendations_data:

            if not isinstance(item, dict):
                continue

            title = str(
                item.get("title", "")
            ).strip()

            reason = str(
                item.get("reason", "")
            ).strip()

            action = str(
                item.get("action", "")
            ).strip()

            # Required fields.
            if not title or not action:
                continue

            priority = str(
                item.get(
                    "priority",
                    "medium",
                )
            ).strip().lower()

            if priority not in {
                "low",
                "medium",
                "high",
            }:
                priority = "medium"

            # ----------------------------------------------------
            # SAFE LIST NORMALIZATION
            # ----------------------------------------------------

            concept_ids = item.get(
                "concept_ids",
                [],
            )

            if not isinstance(
                concept_ids,
                list,
            ):
                concept_ids = []

            concept_ids = [
                str(value).strip()
                for value in concept_ids
                if value is not None
                and str(value).strip()
            ]

            source_types = item.get(
                "source_types",
                [],
            )

            if not isinstance(
                source_types,
                list,
            ):
                source_types = []

            source_types = [
                str(value).strip()
                for value in source_types
                if value is not None
                and str(value).strip()
            ]

            recommendations.append(
                GeneratedRecommendation(
                    title=title,
                    reason=reason,
                    action=action,
                    priority=priority,
                    concept_ids=concept_ids,
                    source_types=source_types,
                )
            )

        # ========================================================
        # EMPTY RESULT
        # ========================================================

        if not recommendations:
            print(
                "[Recommender] AI returned no usable recommendations."
            )

        print(
            "[Recommender] "
            f"usable_recommendations={len(recommendations)}"
        )

        return RecommendationGenerationResult(
            recommendations=recommendations
        )

    @staticmethod
    def _extract_json(text: str) -> str:

        text = text.strip()

        if text.startswith("```"):
            lines = text.splitlines()

            lines = [
                line
                for line in lines
                if not line.strip().startswith("```")
            ]

            text = "\n".join(lines)

        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:
            raise ValueError(
                "AI recommendation response did not contain valid JSON."
            )

        return text[start:end + 1]
